Cell 1 — Mount Drive

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive
# Experiment M2b - Adapt working Tarifit MMS checkpoint
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Cell 2 — Define project paths

In [ ]:
# ============================================================
# CELL 2: Define M2b project paths
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

TRAIN_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1_1"
    / "train.csv"
)

VALIDATION_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1_1"
    / "validation.csv"
)

EXCLUDED_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1_1"
    / "validation_ctc_excluded.csv"
)

M2B_OUTPUT_DIR = (
    PROJECT_ROOT
    / "models"
    / "mms_M2b_tarifit_v1_1"
)

print("Project:", PROJECT_ROOT.exists())
print("Train:", TRAIN_CSV.exists())
print("Validation:", VALIDATION_CSV.exists())
print("Exclusion file:", EXCLUDED_CSV.exists())

Project: True
Train: True
Validation: True
Exclusion file: False


In [ ]:
# ============================================================
# CELL 2B: Recreate CTC exclusion file
# ============================================================

import pandas as pd

BAD_VALIDATION_IDS = [
    "REC138_SEG0025",
    "REC138_SEG0033",
    "REC138_SEG0041",
    "REC138_SEG0042",
    "REC138_SEG0045",
]

validation_df_temp = pd.read_csv(VALIDATION_CSV)

excluded_df = validation_df_temp[
    validation_df_temp["segment_id"].isin(BAD_VALIDATION_IDS)
].copy()

excluded_df["exclusion_reason"] = (
    "audio-transcript pair is CTC-infeasible; "
    "manual inspection required"
)

# Try saving for reproducibility
EXCLUDED_CSV.parent.mkdir(
    parents=True,
    exist_ok=True
)

excluded_df.to_csv(
    EXCLUDED_CSV,
    index=False,
    encoding="utf-8"
)

print("Excluded segments:", len(excluded_df))
print(excluded_df["segment_id"].tolist())
print("Saved:", EXCLUDED_CSV.exists())

Excluded segments: 5
['REC138_SEG0025', 'REC138_SEG0033', 'REC138_SEG0041', 'REC138_SEG0042', 'REC138_SEG0045']
Saved: True


Cell 3 — Install dependencies

In [ ]:
# ============================================================
# CELL 3: Install dependencies
# ============================================================

!pip install -q transformers datasets accelerate jiwer soundfile safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 115.3 MB/s eta 0:00:00


Cell 4 — Load metadata and recreate clean validation

In [ ]:
# ============================================================
# CELL 4: Load train + clean validation metadata
# ============================================================

import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
validation_df = pd.read_csv(VALIDATION_CSV)

excluded_df = pd.read_csv(EXCLUDED_CSV)

excluded_ids = set(
    excluded_df["segment_id"].tolist()
)

validation_clean_df = (
    validation_df[
        ~validation_df["segment_id"].isin(excluded_ids)
    ]
    .reset_index(drop=True)
)

print("Train:", len(train_df))
print("Original validation:", len(validation_df))
print("Clean validation:", len(validation_clean_df))

print(
    "Excluded:",
    len(validation_df) - len(validation_clean_df)
)

Train: 1472
Original validation: 133
Clean validation: 128
Excluded: 5


Cell 5 — Load the working Tarifit MMS checkpoint

In [ ]:
# ============================================================
# CELL 5: Load existing Tarifit MMS checkpoint
# ============================================================

import torch
from transformers import AutoProcessor, Wav2Vec2ForCTC

MODEL_ID = "iukocha/mms-tachebdant-from-tarifit"

processor_original = AutoProcessor.from_pretrained(
    MODEL_ID
)

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_ID
)

print("Model:", MODEL_ID)

print(
    "Parameters:",
    round(model.num_parameters() / 1e6, 1),
    "M"
)

print(
    "Original vocabulary:",
    model.config.vocab_size
)

print(
    "Tokenizer vocabulary:",
    len(processor_original.tokenizer)
)

processor_config.json:   0%|          | 0.00/299 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/56.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.86GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

Model: iukocha/mms-tachebdant-from-tarifit
Parameters: 964.7 M
Original vocabulary: 46
Tokenizer vocabulary: 46


Cell 6 — Inspect MMS adapter configuration

In [ ]:
# ============================================================
# CELL 6: Inspect adapter configuration
# ============================================================

print("add_adapter:")
print(
    getattr(
        model.config,
        "add_adapter",
        "NOT PRESENT"
    )
)

print("\nadapter_attn_dim:")
print(
    getattr(
        model.config,
        "adapter_attn_dim",
        "NOT PRESENT"
    )
)

print("\ntarget_lang:")
print(
    getattr(
        model.config,
        "target_lang",
        "NOT PRESENT"
    )
)

print("\nnum_adapter_layers:")
print(
    getattr(
        model.config,
        "num_adapter_layers",
        "NOT PRESENT"
    )
)

add_adapter:
False

adapter_attn_dim:
16

target_lang:
NOT PRESENT

num_adapter_layers:
3


Cell 7 — Find adapter parameters in the checkpoint

In [ ]:
# ============================================================
# CELL 7: Inspect adapter parameters
# ============================================================

adapter_named_params = [
    (name, param)
    for name, param in model.named_parameters()
    if "adapter" in name.lower()
]

print(
    "Number of parameter tensors containing 'adapter':",
    len(adapter_named_params)
)

adapter_param_count = sum(
    p.numel()
    for _, p in adapter_named_params
)

print(
    "Adapter parameter count:",
    round(adapter_param_count / 1e6, 3),
    "M"
)

print("\nFirst adapter parameter names:")

for name, param in adapter_named_params[:20]:
    print(
        name,
        tuple(param.shape)
    )

Number of parameter tensors containing 'adapter': 288
Adapter parameter count: 2.151 M

First adapter parameter names:
wav2vec2.encoder.layers.0.adapter_layer.norm.weight (1280,)
wav2vec2.encoder.layers.0.adapter_layer.norm.bias (1280,)
wav2vec2.encoder.layers.0.adapter_layer.linear_1.weight (16, 1280)
wav2vec2.encoder.layers.0.adapter_layer.linear_1.bias (16,)
wav2vec2.encoder.layers.0.adapter_layer.linear_2.weight (1280, 16)
wav2vec2.encoder.layers.0.adapter_layer.linear_2.bias (1280,)
wav2vec2.encoder.layers.1.adapter_layer.norm.weight (1280,)
wav2vec2.encoder.layers.1.adapter_layer.norm.bias (1280,)
wav2vec2.encoder.layers.1.adapter_layer.linear_1.weight (16, 1280)
wav2vec2.encoder.layers.1.adapter_layer.linear_1.bias (16,)
wav2vec2.encoder.layers.1.adapter_layer.linear_2.weight (1280, 16)
wav2vec2.encoder.layers.1.adapter_layer.linear_2.bias (1280,)
wav2vec2.encoder.layers.2.adapter_layer.norm.weight (1280,)
wav2vec2.encoder.layers.2.adapter_layer.norm.bias (1280,)
wav2vec2.encode

Cell 8 — Inspect currently trainable parameters

In [ ]:
# ============================================================
# CELL 8: Inspect model parameter groups
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

lm_head_params = sum(
    p.numel()
    for p in model.lm_head.parameters()
)

adapter_params = sum(
    p.numel()
    for name, p in model.named_parameters()
    if "adapter" in name.lower()
)

print(
    "Total:",
    round(total_params / 1e6, 2),
    "M"
)

print(
    "LM head:",
    round(lm_head_params / 1e6, 3),
    "M"
)

print(
    "Adapter-like params:",
    round(adapter_params / 1e6, 3),
    "M"
)

Total: 964.71 M
LM head: 0.059 M
Adapter-like params: 2.151 M


In [ ]:
# ============================================================
# CELL 9: Load Corpus V1.1 target tokenizer
# ============================================================

from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor
)

TARGET_LANG = "rif"

TARGET_VOCAB_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mms_M2_tokenizer_v1_1"
    / "vocab.json"
)

target_tokenizer = Wav2Vec2CTCTokenizer(
    str(TARGET_VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
    target_lang=TARGET_LANG
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

target_processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=target_tokenizer
)

print("Original MMS vocabulary:", len(processor_original.tokenizer))
print("Corpus V1.1 vocabulary:", len(target_tokenizer))

Original MMS vocabulary: 46
Corpus V1.1 vocabulary: 38


In [ ]:
# ============================================================
# CELL 10: Compare original and Corpus V1.1 vocabularies
# ============================================================

source_vocab = processor_original.tokenizer.get_vocab()
target_vocab = target_tokenizer.get_vocab()

print("ORIGINAL MMS:")
print(
    sorted(source_vocab, key=source_vocab.get)
)

print("\nCORPUS V1.1:")
print(
    sorted(target_vocab, key=target_vocab.get)
)

ORIGINAL MMS:
['!', ',', '-', '.', '?', 'a', 'b', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'p', 'q', 'r', 's', 't', 'u', 'w', 'x', 'y', 'z', 'ç', 'ā', 'č', 'ŋ', 'š', 'ə', 'ɛ', 'ɣ', 'ḍ', 'ḥ', 'ṣ', 'ṭ', '[UNK]', 'ẓ', '[PAD]', '<s>', '</s>', ' ']

CORPUS V1.1:
['|', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'p', 'q', 'r', 's', 't', 'u', 'w', 'x', 'y', 'z', 'ǧ', 'ɛ', 'ɣ', 'ʷ', 'ḍ', 'ḥ', 'ṣ', 'ṭ', 'ẓ', '[UNK]', '[PAD]', '<s>', '</s>']


In [ ]:
# ============================================================
# CELL 11: Preserve original Tarifit CTC head
# ============================================================

import copy

original_lm_head = copy.deepcopy(
    model.lm_head
)

print(
    "Original head:",
    original_lm_head.weight.shape
)

Original head: torch.Size([46, 1280])


In [ ]:
# ============================================================
# CELL 12: Transfer Tarifit CTC-head knowledge to Corpus V1.1
# ============================================================

import torch
import torch.nn as nn

hidden_size = model.lm_head.in_features

new_lm_head = nn.Linear(
    hidden_size,
    len(target_tokenizer)
)

# Systematic orthographic correspondences
orthographic_map = {
    "e": "ə",
    "c": "š",
}

copied_tokens = []
random_tokens = []

with torch.no_grad():

    for target_token, target_id in target_vocab.items():

        # Direct correspondence
        if target_token in source_vocab:
            source_token = target_token

        # Known systematic correspondence
        elif (
            target_token in orthographic_map
            and orthographic_map[target_token] in source_vocab
        ):
            source_token = orthographic_map[target_token]

        else:
            random_tokens.append(target_token)
            continue

        source_id = source_vocab[source_token]

        new_lm_head.weight[target_id].copy_(
            original_lm_head.weight[source_id]
        )

        new_lm_head.bias[target_id].copy_(
            original_lm_head.bias[source_id]
        )

        copied_tokens.append(
            (target_token, source_token)
        )


model.lm_head = new_lm_head

model.config.vocab_size = len(target_tokenizer)
model.config.pad_token_id = target_tokenizer.pad_token_id
model.config.ctc_zero_infinity = True

print("New head:", model.lm_head.weight.shape)

print("\nTransferred tokens:")
for pair in copied_tokens:
    print(pair)

print("\nRandomly initialized tokens:")
print(random_tokens)

New head: torch.Size([38, 1280])

Transferred tokens:
('a', 'a')
('b', 'b')
('c', 'š')
('d', 'd')
('e', 'e')
('f', 'f')
('g', 'g')
('h', 'h')
('i', 'i')
('j', 'j')
('k', 'k')
('l', 'l')
('m', 'm')
('n', 'n')
('p', 'p')
('q', 'q')
('r', 'r')
('s', 's')
('t', 't')
('u', 'u')
('w', 'w')
('x', 'x')
('y', 'y')
('z', 'z')
('ɛ', 'ɛ')
('ɣ', 'ɣ')
('ḍ', 'ḍ')
('ḥ', 'ḥ')
('ṣ', 'ṣ')
('ṭ', 'ṭ')
('ẓ', 'ẓ')
('[UNK]', '[UNK]')
('[PAD]', '[PAD]')
('<s>', '<s>')
('</s>', '</s>')

Randomly initialized tokens:
['ǧ', 'ʷ', '|']


In [ ]:
# ============================================================
# CELL 13: Activate only existing adapters + CTC head
# ============================================================

# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the existing Tarifit adapter layers
for name, param in model.named_parameters():
    if "adapter_layer" in name:
        param.requires_grad = True

# Unfreeze new transferred CTC head
for param in model.lm_head.parameters():
    param.requires_grad = True


trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print(
    "Trainable:",
    round(trainable / 1e6, 3),
    "M"
)

print(
    "Total:",
    round(total / 1e6, 2),
    "M"
)

print(
    "Percentage:",
    round(100 * trainable / total, 4),
    "%"
)

Trainable: 2.2 M
Total: 964.7 M
Percentage: 0.228 %


In [ ]:
# ============================================================
# CELL 14: Build clean M2b train/validation datasets
# ============================================================

from datasets import Dataset, DatasetDict

train_dataset_df = train_df.copy()

validation_dataset_df = validation_clean_df.copy()

dataset_m2b = DatasetDict({
    "train": Dataset.from_pandas(
        train_dataset_df,
        preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        validation_dataset_df,
        preserve_index=False
    )
})

print(dataset_m2b)

print("Train:", len(dataset_m2b["train"]))
print("Validation:", len(dataset_m2b["validation"]))

DatasetDict({
    train: Dataset({
        features: ['segment_id', 'recording_id', 'speaker_group_id', 'audio_path', 'duration_seconds', 'transcription', 'absolute_audio_path'],
        num_rows: 1472
    })
    validation: Dataset({
        features: ['segment_id', 'recording_id', 'speaker_group_id', 'audio_path', 'duration_seconds', 'transcription', 'absolute_audio_path'],
        num_rows: 128
    })
})
Train: 1472
Validation: 128


In [ ]:
# ============================================================
# CELL 15: Define M2b audio/text preprocessing
# ============================================================

import soundfile as sf

def prepare_m2b_dataset(example):

    audio_file = PROJECT_ROOT / example["audio_path"]

    audio, sampling_rate = sf.read(audio_file)

    inputs = target_processor(
        audio,
        sampling_rate=sampling_rate
    )

    example["input_values"] = inputs.input_values[0]

    example["input_length"] = len(
        example["input_values"]
    )

    example["labels"] = target_tokenizer(
        example["transcription"]
    ).input_ids

    return example


print("M2b preprocessing ready.")

M2b preprocessing ready.


In [ ]:
# ============================================================
# CELL 16: Preprocess full M2b dataset
# ============================================================

m2b_dataset = dataset_m2b.map(
    prepare_m2b_dataset,
    remove_columns=dataset_m2b["train"].column_names,
    num_proc=1
)

print(m2b_dataset)

Map:   0%|          | 0/1472 [00:00<?, ? examples/s]

Map:   0%|          | 0/128 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 1472
    })
    validation: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 128
    })
})


In [ ]:
# ============================================================
# CELL 17: Define M2b CTC collator
# ============================================================

from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch


@dataclass
class DataCollatorCTCWithPadding:

    processor: Any
    padding: Union[bool, str] = True

    def __call__(self, features):

        input_features = [
            {"input_values": f["input_values"]}
            for f in features
        ]

        label_features = [
            {"input_ids": f["labels"]}
            for f in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=target_processor
)

print("M2b collator ready.")

M2b collator ready.


In [ ]:
# ============================================================
# CELL 18: Define M2b WER/CER
# ============================================================

import numpy as np
from jiwer import wer, cer


def compute_metrics(pred):

    pred_ids = np.argmax(
        pred.predictions,
        axis=-1
    )

    label_ids = pred.label_ids.copy()

    label_ids[label_ids == -100] = (
        target_processor.tokenizer.pad_token_id
    )

    pred_str = target_processor.batch_decode(
        pred_ids
    )

    label_str = target_processor.batch_decode(
        label_ids,
        group_tokens=False
    )

    return {
        "wer": wer(label_str, pred_str) * 100,
        "cer": cer(label_str, pred_str) * 100
    }


print("Metrics ready.")

Metrics ready.


In [ ]:
# ============================================================
# CELL 19: Configure MMS M2b fine-tuning
# ============================================================

from transformers import TrainingArguments

M2B_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

training_args = TrainingArguments(
    output_dir=str(M2B_OUTPUT_DIR),

    # T4-safe batching
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=1,

    # Gentle adaptation
    learning_rate=2e-4,
    num_train_epochs=8,
    warmup_steps=20,

    # GPU
    fp16=True,

    # Evaluation
    eval_strategy="epoch",

    # Checkpointing
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # Logging
    logging_strategy="steps",
    logging_steps=25,

    report_to="none",
    seed=42,
)

print("M2b training arguments ready.")

M2b training arguments ready.


In [ ]:
# ============================================================
# CELL 20: Create M2b Trainer
# ============================================================

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=m2b_dataset["train"],
    eval_dataset=m2b_dataset["validation"],

    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("M2b Trainer ready.")

M2b Trainer ready.


In [ ]:
# ============================================================
# CELL 21: Final sanity check before M2b training
# ============================================================

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Train examples:", len(m2b_dataset["train"]))
print("Validation examples:", len(m2b_dataset["validation"]))
print("Tokenizer size:", len(target_tokenizer))
print("Model vocabulary:", model.config.vocab_size)
print("Trainable parameters:", round(trainable_params / 1e6, 3), "M")
print("Learning rate:", training_args.learning_rate)
print("Epochs:", training_args.num_train_epochs)
print("CTC zero infinity:", model.config.ctc_zero_infinity)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Train examples: 1472
Validation examples: 128
Tokenizer size: 38
Model vocabulary: 38
Trainable parameters: 2.2 M
Learning rate: 0.0002
Epochs: 8
CTC zero infinity: True
CUDA: True
GPU: Tesla T4


In [ ]:
# ============================================================
# CELL 22: Start MMS M2b fine-tuning
# ============================================================

train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Wer,Cer
1,5.887029,3.117467,84.891486,42.662144
2,5.700375,2.953110,87.145242,43.195218
3,4.524546,3.125656,88.105175,43.768678


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]